Purpose of this notebook:
- Check out delta share dataset for Zillow properties

In [0]:
# import libraries
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
import plotly.express as px
import pandas as pd

In [0]:
# check what the dataset looks like

%sql
SELECT * 
FROM bright_data_zillow_properties_information_dataset.datasets.zillow_properties

In [0]:
# light transformations

# Define the schema for the JSON columns
address_schema = StructType([
    StructField("street", StringType(), True),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("zipcode", StringType(), True)
])

price_history_schema = StructType([
    StructField("date", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("event", StringType(), True)
])

# Load the Zillow dataset into a DataFrame
zillow_df = spark.table("bright_data_zillow_properties_information_dataset.datasets.zillow_properties")

# Parse the JSON columns
zillow_df = zillow_df.withColumn("parsed_address", from_json(col("ADDRESS"), address_schema))
zillow_df = zillow_df.withColumn("parsed_price_history", from_json(col("PRICEHISTORY"), price_history_schema))

# Select the necessary columns including the parsed JSON columns
zillow_df = zillow_df.select(
    col("TIMESTAMP"),
    col("ZPID"),
    col("CITY"),
    col("STATE"),
    col("HOMESTATUS"),
    col("parsed_address.street").alias("street"),
    col("parsed_address.city").alias("parsed_city"),
    col("parsed_address.state").alias("parsed_state"),
    col("parsed_address.zipcode").alias("parsed_zipcode"),
    col("BEDROOMS"),
    col("BATHROOMS"),
    col("PRICE"),
    col("YEARBUILT"),
    col("STREETADDRESS"),
    col("ZIPCODE"),
    col("LONGITUDE"),
    col("LATITUDE"),
    col("HOMETYPE"),
    col("LIVINGAREAVALUE"),
    col("LIVINGAREAUNITSSHORT"),
    col("RENTZESTIMATE"),
    col("CURRENCY"),
    col("COUNTRY"),
    col("HDPURL"),
    col("LASTSOLDPRICE"),
    col("LIVINGAREAUNITS"),
    col("DESCRIPTION"),
    col("parsed_price_history.date").alias("price_history_date"),
    col("parsed_price_history.price").alias("price_history_price"),
    col("parsed_price_history.event").alias("price_history_event"),
    col("BROKERAGENAME"),
    col("HDPTYPEDIMENSION"),
    col("TIMEZONE"),
    col("PHOTOS"),
    col("URL")
)

display(zillow_df)

In [0]:
# Convert column names to lowercase out of preference

zillow_df = zillow_df.toDF(*[c.lower() for c in zillow_df.columns])
display(zillow_df)

In [0]:
# Write cleaned data to delta table

zillow_df.write.format("delta").mode("overwrite").saveAsTable("workspace.zillow.cleaned_data")

In [0]:
# Map time!!

# Convert Spark DataFrame to Pandas DataFrame
zillow_pd_df = zillow_df.select("latitude", "longitude", "price", "hometype").toPandas()

# Ensure latitude, longitude, and price columns are numeric
zillow_pd_df['latitude'] = pd.to_numeric(zillow_pd_df['latitude'], errors='coerce')
zillow_pd_df['longitude'] = pd.to_numeric(zillow_pd_df['longitude'], errors='coerce')
zillow_pd_df['price'] = pd.to_numeric(zillow_pd_df['price'], errors='coerce')

# Drop rows with NaN values in latitude, longitude, or price
zillow_pd_df = zillow_pd_df.dropna(subset=['latitude', 'longitude', 'price'])

# Create a scatter mapbox plot
fig = px.scatter_mapbox(
    zillow_pd_df,
    lat="latitude",
    lon="longitude",
    size="price",
    color="hometype",
    hover_name="price",
    hover_data={"latitude": False, "longitude": False},
    zoom=10,
    mapbox_style="carto-positron"
)

# Display the map
display(fig)